# Phase 2 Partial Coherence

Study-grade source-shape comparison for point, dipole, quadrupole, and annular illumination.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().resolve()
if not (repo / "src").exists() and (repo.parent / "src").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

In [ ]:
from src import constants as C
from src.aerial import contrast
from src.illuminator import (
    annular_source,
    dipole_source,
    partial_coherent_aerial_image,
    point_source,
    quadrupole_source,
)
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.pupil import PupilSpec

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
pattern = line_space_pattern(grid, pitch_m=40e-9, duty_cycle=0.5)
mask_field = kirchhoff_mask(pattern)
pupil = PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0)

sources = (
    point_source(name="point"),
    dipole_source(0.65, orientation="x", name="dipole x"),
    quadrupole_source(0.55, name="quadrupole"),
    annular_source(0.30, 0.75, num_radial=2, num_azimuthal=12, name="annular"),
)

In [ ]:
results = {}
contrasts = {}

for source in sources:
    result = partial_coherent_aerial_image(
        mask_field,
        grid,
        source,
        pupil,
        anamorphic=False,
    )
    cy = result.wafer_grid.ny // 2
    results[source.name] = result
    contrasts[source.name] = contrast(result.intensity[cy, :])

contrasts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

for name, result in results.items():
    cy = result.wafer_grid.ny // 2
    x_nm = (np.arange(result.wafer_grid.nx) - result.wafer_grid.nx // 2) * result.wafer_grid.pixel_x_m * 1e9
    axes[0].plot(x_nm, result.intensity[cy, :], label=name)

axes[0].set_xlabel("x at wafer (nm)")
axes[0].set_ylabel("normalized intensity")
axes[0].set_title("Center-line aerial image")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].bar(list(contrasts.keys()), list(contrasts.values()), color=["#4C78A8", "#F58518", "#54A24B", "#B279A2"])
axes[1].set_ylim(0.0, 1.05)
axes[1].set_ylabel("Michelson contrast")
axes[1].set_title("Source-shape contrast trend")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.25)

plt.show()